# 03. 양자화 + 대회 채점 + 속도 측정

02에서 고른 모델로 **최적화 전 / 후**를 비교하는 표를 만들어요. 예선 보고서의 핵심 결과예요.

| 모델 | 설명 |
| --- | --- |
| FP32 | 최적화 전 원본 (베이스라인) |
| INT8 (전체) | 모든 계산을 INT8로 |
| INT8 (헤드 FP32) | 박스 계산 부분만 FP32로 남기고 나머지는 INT8 (혼합 정밀도) |

재는 것: **KITTI Moderate AP40** (대회 공식 점수), 유지율, 파라미터 수, FLOPs, 모델 크기, CPU latency, 메모리

- GPU 없이 CPU 런타임으로도 돌아가요. 대략 20~30분 걸려요.
- 시험지(eval) 1,000장은 **여기서 처음이자 마지막으로** 써요.

## 1. Google Drive 연결 + 데이터 준비 + 도구 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/embeded_ondeviceAI/runs'

In [ ]:
%%bash
set -e   # 중간에 실패하면 바로 멈추고 어디서 실패했는지 보여 줌
git clone -q https://github.com/gyu-young-04/embeded_ondeviceAI.git /content/embeded_ondeviceAI 2>/dev/null || git -C /content/embeded_ondeviceAI pull -q
mkdir -p /content/kitti && cd /content/kitti
for f in data_object_label_2.zip data_object_image_2.zip; do
  wget -c -q https://s3.eu-central-1.amazonaws.com/avg-kitti/$f
  unzip -q -n $f 'training/*'
done
[ "$(ls training/image_2 | wc -l)" = 7481 ] || { echo '사진이 7481장이 아니에요. /content/kitti 를 지우고 다시 실행하세요.'; exit 1; }
cd /content/embeded_ondeviceAI
[ -f /content/kitti_yolo/kitti.yaml ] || python scripts/prepare_kitti.py --kitti /content/kitti --out /content/kitti_yolo

In [ ]:
!pip -q install ultralytics onnx onnxruntime onnxslim
import sys
sys.path.insert(0, '/content/embeded_ondeviceAI/scripts')
import ultralytics, onnxruntime, torch
print('ultralytics', ultralytics.__version__, '| onnxruntime', onnxruntime.__version__, '| torch', torch.__version__)

## 2. 어떤 모델을 최적화할지 고르기

02 노트북 비교표에서 제일 좋았던 실험 이름을 넣어요.

In [ ]:
NAME = 'yolov8n_960_e20'

from pathlib import Path
run = Path(RUNS) / NAME
best_pt = run / 'weights' / 'best.pt'
assert best_pt.exists(), f'{best_pt} 가 없어요. 02 노트북에서 학습이 끝났는지 확인하세요.'

IMGSZ = int(NAME.split('_')[1])
W = IMGSZ
H = round(IMGSZ * 375 / 1242 / 32) * 32   # KITTI 비율에 맞춘 32의 배수 (960 → 288)
OUT = run / 'optimize'
OUT.mkdir(exist_ok=True)
print(f'입력 크기: {W} x {H}, 결과 폴더: {OUT}')

## 3. ONNX로 변환 (FP32 베이스라인)

입력 크기를 고정해서(static shape) 내보내요. Hailo 컴파일러도 고정 크기를 써요.

In [ ]:
import shutil
from ultralytics import YOLO

fp32 = OUT / 'fp32.onnx'
if not fp32.exists():
    f = YOLO(str(best_pt)).export(format='onnx', imgsz=[H, W], opset=13, simplify=True, dynamic=False)
    shutil.copy(f, fp32)
print(fp32)

## 4. INT8 양자화 (PTQ)

- Calibration(눈금 맞추기)에는 **학습(train) 사진 300장**을 써요. 시험지는 쓰지 않아요.
- 두 가지를 만들어서 비교해요: 전체 INT8 / 박스 계산 부분만 FP32.

In [ ]:
%cd /content/embeded_ondeviceAI
!python scripts/quantize_onnx.py --model {OUT}/fp32.onnx --out {OUT}/int8_full.onnx --calib-list /content/kitti_yolo/train.txt
!python scripts/quantize_onnx.py --model {OUT}/fp32.onnx --out {OUT}/int8_head_fp32.onnx --calib-list /content/kitti_yolo/train.txt --keep-head-fp32

## 5. 대회 채점 (KITTI Moderate AP40, 시험지 1,000장)

모델마다 1,000장을 추론해서 KITTI 형식 예측 파일을 만들고 공식 방식으로 채점해요. 모델당 몇 분 걸려요.

In [ ]:
import onnx_infer, kitti_eval

MODELS = {'FP32': 'fp32.onnx', 'INT8 (전체)': 'int8_full.onnx', 'INT8 (헤드 FP32)': 'int8_head_fp32.onnx'}
ids = [l.strip() for l in open('/content/embeded_ondeviceAI/eval_val.txt') if l.strip()]
scores = {}
for label, fname in MODELS.items():
    pred_dir = f'/content/preds/{NAME}/{fname[:-5]}'
    onnx_infer.predict_folder(OUT / fname, '/content/kitti/training/image_2', ids, pred_dir)
    result, mod_map = kitti_eval.evaluate(pred_dir, '/content/kitti/training/label_2', ids)
    scores[label] = (result, mod_map)
    print(f'[{label}]')
    print(kitti_eval.format_result(result, mod_map), '\n')

## 6. 속도 · 메모리 측정 (CPU, batch 1)

- 같은 컴퓨터에서 warm-up 10번 후 100번 재서 평균을 내요. 모델 실행 시간만 재요 (전처리·NMS 제외).
- 모델마다 새로 프로그램을 띄워서 재야 메모리 값이 섞이지 않아요.

In [ ]:
import json, os, subprocess

threads = os.cpu_count()
bench = {}
for label, fname in MODELS.items():
    out = subprocess.run(['python', 'scripts/benchmark_onnx.py', '--model', str(OUT / fname), '--threads', str(threads)],
                         capture_output=True, text=True, check=True).stdout
    bench[label] = json.loads(out.strip().splitlines()[-1])
    print(label, bench[label])
!lscpu | grep 'Model name'

## 7. 최종 비교표

이 표가 보고서의 **Baseline 대비 정량 비교 결과**예요. Drive에 `results.csv` 로도 저장돼요.

In [ ]:
import pandas as pd
from ultralytics.utils.torch_utils import get_flops

torch_model = YOLO(str(best_pt)).model
params = sum(p.numel() for p in torch_model.parameters())
gflops = get_flops(torch_model, [H, W])
base = scores['FP32'][1]

rows = []
for label, fname in MODELS.items():
    result, mod_map = scores[label]
    b = bench[label]
    rows.append({
        '모델': label,
        'Car': round(result['Car'][1], 2),
        'Pedestrian': round(result['Pedestrian'][1], 2),
        'Cyclist': round(result['Cyclist'][1], 2),
        'Moderate mAP': round(mod_map, 2),
        '유지율(%)': round(mod_map / base * 100, 1),
        '파라미터(M)': round(params / 1e6, 2),
        'GFLOPs': round(gflops, 2),
        '파일 크기(MB)': round((OUT / fname).stat().st_size / 1e6, 1),
        'latency(ms)': f"{b['latency_ms_mean']} ± {b['latency_ms_std']}",
        'latency 중앙값(ms)': b['latency_ms_p50'],
        'FPS': b['fps'],
        '메모리(MB)': b['model_mem_mb'],
    })

table = pd.DataFrame(rows)
table.to_csv(OUT / 'results.csv', index=False)
print(f'{NAME} | 입력 {W}x{H} | CPU {threads}스레드 | KITTI Moderate AP40 (eval_val 1,000장)')
table

### 표 읽는 법

- **유지율**: 원본 점수의 몇 %를 지켰는지. 100%에 가까울수록 좋아요.
- **파라미터 / GFLOPs**: 양자화는 숫자의 길이만 줄이고 개수는 그대로라서 세 모델 모두 같아요.
- **파일 크기, latency**: 작을수록 좋아요. INT8은 보통 크기 약 1/4, 속도 약 2배.
- **INT8 (전체)** 점수가 크게 떨어지고 **INT8 (헤드 FP32)** 는 괜찮다면, "박스 좌표와 점수를 한 숫자 범위로 줄이면 점수가 뭉개져서, 그 부분만 FP32로 남겼다"가 보고서의 핵심 분석 포인트가 돼요.